In [1]:
# ================================================
# SISTEM REKOMENDASI BERITA (TF-IDF + CLUSTER)
# ================================================

# 1. Import Library
import pandas as pd
import numpy as np
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

# NLP Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from tqdm.notebook import tqdm
tqdm.pandas()

# ================================================
# 2. Load Dataset
# ================================================
df = pd.read_csv("politik_merge.csv")
print("Jumlah data awal:", len(df))

# ================================================
# 3. Sampling Dataset → 499 artikel
# (supaya stemming cepat & sesuai narasi PI)
# ================================================
if len(df) > 499:
    df = df.sample(n=499, random_state=42).reset_index(drop=True)
print("Jumlah data setelah sampling:", len(df))

# ================================================
# 4. Preprocessing Lengkap
# ================================================
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

def clean_text(text):
    if isinstance(text, str):
        text = text.lower()                        # case folding
        text = re.sub(r'\d+', '', text)            # hapus angka
        text = re.sub(r'\W+', ' ', text)           # hapus simbol
        text = re.sub(r'\s+', ' ', text).strip()   # hapus spasi
        tokens = text.split()
        tokens = [t for t in tokens if t not in stopwords]   # stopword removal
        text = ' '.join([stemmer.stem(t) for t in tokens])   # stemming
        return text
    return ""

df["cleaned"] = df["Content"].astype(str).apply(clean_text)

# Gabungkan judul + konten
df["combined"] = df["Judul"].astype(str) + " " + df["cleaned"]

print("\nContoh teks setelah preprocessing:\n")
print(df[["Judul", "combined"]].head())

# ================================================
# 5. Clustering dengan KMeans
# ================================================
vectorizer_cluster = TfidfVectorizer(max_features=3000)
X_cluster = vectorizer_cluster.fit_transform(df["combined"])

kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_cluster)

print("\nDistribusi cluster:")
print(df["Cluster"].value_counts())

# Simpan dataset final
df.to_csv("news_with_cluster.csv", index=False)
print("\n✅ File 'news_with_cluster.csv' berhasil disimpan!")
print("Jumlah data akhir (clustered):", len(df))

# ================================================
# 6. TF-IDF Vectorization (untuk rekomendasi)
# ================================================
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,3)   # unigram, bigram, trigram
)
tfidf_matrix = vectorizer.fit_transform(df["combined"])
print("\nTF-IDF matrix shape:", tfidf_matrix.shape)

# ================================================
# 7. Simpan Model & Data
# ================================================
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))
pickle.dump(tfidf_matrix, open("tfidf_matrix.pkl", "wb"))

print("\n✅ File model berhasil disimpan: tfidf_vectorizer.pkl & tfidf_matrix.pkl")

# ================================================
# 8. Fungsi Rekomendasi
# ================================================
def get_recommendations(title, top_n=5):
    if title not in df["Judul"].values:
        return ["Judul tidak ditemukan"]

    idx = df[df["Judul"] == title].index[0]
    cluster_id = df.iloc[idx]["Cluster"]

    # subset artikel dalam cluster yang sama
    subset_idx = df[df["Cluster"] == cluster_id].index
    subset_matrix = tfidf_matrix[subset_idx]

    # cosine similarity
    cosine_sim = cosine_similarity(tfidf_matrix[idx], subset_matrix).flatten()

    # urutkan
    similar_idx = cosine_sim.argsort()[::-1]

    # ambil top_n (skip dirinya sendiri)
    similar_idx = similar_idx[1:top_n+1]
    rekomendasi_idx = subset_idx[similar_idx]

    return df.iloc[rekomendasi_idx][["Judul", "Cluster"]]

# ================================================
# 9. Uji Coba Rekomendasi
# ================================================
contoh_judul = df["Judul"].iloc[0]
print("\nJudul dipilih:", contoh_judul)
print("\nRekomendasi:")
print(get_recommendations(contoh_judul, top_n=5))


Jumlah data awal: 45781
Jumlah data setelah sampling: 499

Contoh teks setelah preprocessing:

                                               Judul  \
0  Dalih Kepepet Jambret di Jaksel yang Tertangka...   
1  Mantan Ketum Pemuda Muhammadiyah Sunanto Ditun...   
2  Inilah Jakarta International Stadium atau JIS,...   
3  Tanggapan TPN dan TKN soal Rencana Mahfud Md M...   
4  Meski Diwarnai Ricuh, Polisi Sebut Demo Tolak ...   

                                            combined  
0  Dalih Kepepet Jambret di Jaksel yang Tertangka...  
1  Mantan Ketum Pemuda Muhammadiyah Sunanto Ditun...  
2  Inilah Jakarta International Stadium atau JIS,...  
3  Tanggapan TPN dan TKN soal Rencana Mahfud Md M...  
4  Meski Diwarnai Ricuh, Polisi Sebut Demo Tolak ...  


C:\Users\user\AppData\Roaming\Python\Python312\site-packages\joblib\externals\loky\backend\context.py:150: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\user\AppData\Roaming\Python\Python312\site-packages\joblib\externals\loky\backend\context.py", line 227, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Python312\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro


Distribusi cluster:
Cluster
8    113
1     70
0     69
5     64
6     48
7     42
4     27
2     26
9     23
3     17
Name: count, dtype: int64

✅ File 'news_with_cluster.csv' berhasil disimpan!
Jumlah data akhir (clustered): 499

TF-IDF matrix shape: (499, 10000)

✅ File model berhasil disimpan: tfidf_vectorizer.pkl & tfidf_matrix.pkl

Judul dipilih: Dalih Kepepet Jambret di Jaksel yang Tertangkap gegara Kejebak Macet

Rekomendasi:
                                                 Judul  Cluster
446  Diteriaki 'Maling', Jambret Ponsel Pelari di C...        5
282  Pria di Kalsel Bunuh Teman karena Korban Kedip...        5
6    Siswi SMP Trauma Diculik-Ditodong Cutter: Mama...        5
8    Rampok Ponsel dan Perhiasan Siswi SMP, Pelaku ...        5
143  Warga Sebut Mobil Nyangkut di Separator Margon...        5
